# Sesión 2 (parte 2): Pipelines basados en eventos

## Módulo 4: Event-driven, Eventarc, idempotencia, retry y observabilidad

### Objetivos de este notebook
1. Configurar **notificaciones GCS → Pub/Sub** (la versión más práctica de event-driven sin desplegar Cloud Functions)
2. Subir un archivo CSV de Personio a GCS y **consumir el evento** desde Python
3. Implementar el **patrón de pipeline event-driven** (lectura → transformación → carga idempotente a BQ)
4. Aplicar **retry con backoff exponencial** al cargar a BigQuery
5. Validar registros con **split valid/invalid** y dead letter table
6. Emitir **logs estructurados (JSON)** legibles por Cloud Logging
7. Limpiar todos los recursos creados

**Prerrequisito:** Notebooks de Módulos 1, 2 y 3 ejecutados (datasets `bronze_personio`/`silver_personio`/`gold_people_analytics`, bucket `*-datalake` y conocimiento de Pub/Sub).

> **Nota didáctica:** Eventarc + Cloud Functions sería el camino productivo. En clase usamos GCS Pub/Sub notifications + un consumidor Python para que el pipeline sea **runnable end-to-end en 2-3 minutos** sin esperar deploys de Cloud Functions (que requieren Artifact Registry, build y arranque en frío).

---
## 1. Setup del entorno híbrido

In [1]:
# Instalar dependencias (descomentar en primera ejecución)
# !pip install google-cloud-pubsub google-cloud-bigquery google-cloud-storage pandas pyarrow db-dtypes python-dotenv

import os
import json
import time
import logging
import sys
import re
from datetime import datetime, timezone
from io import StringIO
from functools import wraps
import pandas as pd
from google.cloud import bigquery, storage, pubsub_v1
from google.api_core.exceptions import NotFound, AlreadyExists, GoogleAPICallError, PermissionDenied
import warnings
warnings.filterwarnings('ignore')

# --- Detección de entorno ---
IN_VERTEX_AI = any([
    os.environ.get("DL_ANACONDA_HOME"),
    os.path.exists("/opt/deeplearning/metadata"),
    os.environ.get("GOOGLE_CLOUD_PROJECT"),
])

if IN_VERTEX_AI:
    PROJECT_ID = os.environ.get("GOOGLE_CLOUD_PROJECT", "people-analytics-formacion")
    bq_client = bigquery.Client(project=PROJECT_ID)
    gcs_client = storage.Client(project=PROJECT_ID)
    publisher = pubsub_v1.PublisherClient()
    subscriber = pubsub_v1.SubscriberClient()
else:
    from dotenv import load_dotenv
    from google.oauth2 import service_account
    load_dotenv()
    PROJECT_ID = os.environ.get("GCP_PROJECT_ID", "people-analytics-formacion")
    creds_path = os.environ.get("GOOGLE_APPLICATION_CREDENTIALS", "service-account.json")
    if os.path.exists(creds_path):
        credentials = service_account.Credentials.from_service_account_file(creds_path)
        bq_client = bigquery.Client(project=PROJECT_ID, credentials=credentials)
        gcs_client = storage.Client(project=PROJECT_ID, credentials=credentials)
        publisher = pubsub_v1.PublisherClient(credentials=credentials)
        subscriber = pubsub_v1.SubscriberClient(credentials=credentials)
    else:
        bq_client = bigquery.Client(project=PROJECT_ID)
        gcs_client = storage.Client(project=PROJECT_ID)
        publisher = pubsub_v1.PublisherClient()
        subscriber = pubsub_v1.SubscriberClient()

REGION = "europe-southwest1"
BUCKET_NAME = os.environ.get("GCS_BUCKET_NAME", f"{PROJECT_ID}-datalake")

# Recursos del notebook
TOPIC_GCS = "gcs-personio-events"
SUB_GCS = "gcs-personio-consumer"
EVENT_DRIVEN_PREFIX = "event-driven-demo/"
TABLE_RAW = f"{PROJECT_ID}.bronze_personio.event_driven_raw"
TABLE_ERRORS = f"{PROJECT_ID}.bronze_personio.event_driven_errors"

# Flag que indica si la notificación GCS→Pub/Sub está operativa.
# Si no tenemos permisos para conceder roles/pubsub.publisher al SA de GCS,
# el notebook cae en modo "evento simulado" — el resto del pipeline sigue siendo real.
NOTIFICATION_DISPONIBLE = True

print(f"Proyecto: {PROJECT_ID}")
print(f"Region:   {REGION}")
print(f"Bucket:   {BUCKET_NAME}")
print(f"Topic:    {TOPIC_GCS}")
print(f"Tabla raw:    {TABLE_RAW}")
print(f"Tabla errors: {TABLE_ERRORS}")

Proyecto: project-9176af0b-ecb3-4050-859
Region:   europe-southwest1
Bucket:   project-9176af0b-ecb3-4050-859-datalake
Topic:    gcs-personio-events
Tabla raw:    project-9176af0b-ecb3-4050-859.bronze_personio.event_driven_raw
Tabla errors: project-9176af0b-ecb3-4050-859.bronze_personio.event_driven_errors


### 1.1 Habilitar las APIs necesarias

Pub/Sub y Eventarc deben estar habilitadas. Habilitarlas es **idempotente** — si ya lo están, no hace nada. Tras habilitarlas hay que esperar ~30 segundos para propagación.

In [2]:
# Habilitar APIs necesarias para este notebook (idempotente)
APIS_REQUERIDAS = [
    "pubsub.googleapis.com",          # topics, subscriptions, GCS notifications
    "eventarc.googleapis.com",        # solo si se desplegaran triggers gestionados
    "bigquery.googleapis.com",
    "storage.googleapis.com",
]

import subprocess

for api in APIS_REQUERIDAS:
    print(f"Habilitando {api} ...", end=" ")
    result = subprocess.run(
        ["gcloud", "services", "enable", api, f"--project={PROJECT_ID}"],
        capture_output=True, text=True,
    )
    if result.returncode == 0:
        print("OK")
    else:
        print(f"FALLO\n  stderr: {result.stderr.strip()[:300]}")
        print(f"  → Habilítala manualmente: https://console.cloud.google.com/apis/library/{api}?project={PROJECT_ID}")

print("\nEsperando 30s para propagación...")
time.sleep(30)
print("Listo. Continúa con la siguiente celda.")

Habilitando pubsub.googleapis.com ... OK
Habilitando eventarc.googleapis.com ... OK
Habilitando bigquery.googleapis.com ... OK
Habilitando storage.googleapis.com ... OK

Esperando 30s para propagación...
Listo. Continúa con la siguiente celda.


---
## 2. Configurar notificaciones GCS → Pub/Sub

**Patrón:** cuando un objeto nuevo aparece en `gs://*-datalake/event-driven-demo/`, GCS publica automáticamente un evento en el topic `gcs-personio-events`.

Es el equivalente más simple a un trigger de Eventarc para GCS, **sin necesidad de desplegar Cloud Function ni Cloud Run**.

**Mismo patrón en producción:** un Eventarc trigger entregaría el evento directamente a una Cloud Function. Aquí lo consumimos desde el notebook (subscriber pull) para mantenerlo runnable.

In [3]:
# 1. Asegurarse de que el topic existe
topic_path = publisher.topic_path(PROJECT_ID, TOPIC_GCS)
try:
    publisher.create_topic(request={"name": topic_path})
    print(f"Topic creado: {TOPIC_GCS}")
except AlreadyExists:
    print(f"Topic ya existe: {TOPIC_GCS}")

# 2. Conceder al Service Agent de GCS permiso para publicar en el topic
#    Sin esto, GCS no puede entregar notificaciones al topic de Pub/Sub.
#    El SA de GCS tiene formato fijo: service-{PROJECT_NUMBER}@gs-project-accounts.iam.gserviceaccount.com
#
#    Requiere roles/pubsub.admin (o roles/owner). roles/editor NO es suficiente:
#    incluye pubsub.* pero NO pubsub.topics.getIamPolicy/setIamPolicy.

result = subprocess.run(
    ["gcloud", "projects", "describe", PROJECT_ID, "--format=value(projectNumber)"],
    capture_output=True, text=True, check=True,
)
PROJECT_NUMBER = result.stdout.strip()
GCS_SA = f"service-{PROJECT_NUMBER}@gs-project-accounts.iam.gserviceaccount.com"
GCS_SA_MEMBER = f"serviceAccount:{GCS_SA}"
print(f"GCS Service Agent: {GCS_SA}")

try:
    policy = publisher.get_iam_policy(request={"resource": topic_path})

    ya_tiene_publisher = any(
        b.role == "roles/pubsub.publisher" and GCS_SA_MEMBER in b.members
        for b in policy.bindings
    )

    if ya_tiene_publisher:
        print(f"  El SA de GCS ya tiene roles/pubsub.publisher en el topic")
    else:
        binding = next((b for b in policy.bindings if b.role == "roles/pubsub.publisher"), None)
        if binding is None:
            from google.iam.v1 import policy_pb2
            binding = policy_pb2.Binding(role="roles/pubsub.publisher", members=[GCS_SA_MEMBER])
            policy.bindings.append(binding)
        else:
            binding.members.append(GCS_SA_MEMBER)

        publisher.set_iam_policy(request={"resource": topic_path, "policy": policy})
        print(f"  Concedido roles/pubsub.publisher al SA de GCS sobre el topic")
        print(f"  Esperando 10s para propagación de IAM...")
        time.sleep(10)

except PermissionDenied:
    NOTIFICATION_DISPONIBLE = False
    print(f"\n  [AVISO] No tienes permisos para gestionar IAM del topic.")
    print(f"  roles/editor NO incluye pubsub.topics.getIamPolicy/setIamPolicy.")
    print(f"  Un admin (roles/pubsub.admin o roles/owner) debe ejecutar UNA SOLA VEZ:\n")
    print(f"    gcloud pubsub topics add-iam-policy-binding {TOPIC_GCS} \\")
    print(f"        --member='{GCS_SA_MEMBER}' \\")
    print(f"        --role='roles/pubsub.publisher' \\")
    print(f"        --project={PROJECT_ID}\n")
    print(f"  Sin ese permiso, el notebook se ejecuta en MODO SIMULADO:")
    print(f"  el resto del pipeline (pipeline, idempotencia, retry, validación, logs) es real;")
    print(f"  solo se simula el evento que GCS publicaría al subir el archivo.")

Topic creado: gcs-personio-events
GCS Service Agent: service-221134007818@gs-project-accounts.iam.gserviceaccount.com

  [AVISO] No tienes permisos para gestionar IAM del topic.
  roles/editor NO incluye pubsub.topics.getIamPolicy/setIamPolicy.
  Un admin (roles/pubsub.admin o roles/owner) debe ejecutar UNA SOLA VEZ:

    gcloud pubsub topics add-iam-policy-binding gcs-personio-events \
        --member='serviceAccount:service-221134007818@gs-project-accounts.iam.gserviceaccount.com' \
        --role='roles/pubsub.publisher' \
        --project=project-9176af0b-ecb3-4050-859

  Sin ese permiso, el notebook se ejecuta en MODO SIMULADO:
  el resto del pipeline (pipeline, idempotencia, retry, validación, logs) es real;
  solo se simula el evento que GCS publicaría al subir el archivo.


In [4]:
# 3. Crear la notificación en el bucket (idempotente)
bucket = gcs_client.bucket(BUCKET_NAME)

if NOTIFICATION_DISPONIBLE:
    # Borrar notificaciones previas con mismo prefijo para evitar duplicados
    for n in bucket.list_notifications():
        if n.topic_name == TOPIC_GCS and n.blob_name_prefix == EVENT_DRIVEN_PREFIX:
            n.delete()
            print(f"  Notificación previa borrada: id={n.notification_id}")

    # Crear nueva notificación: solo OBJECT_FINALIZE (cuando se sube un archivo)
    notification = bucket.notification(
        topic_name=TOPIC_GCS,
        event_types=["OBJECT_FINALIZE"],
        blob_name_prefix=EVENT_DRIVEN_PREFIX,
        payload_format="JSON_API_V1",
    )
    try:
        notification.create()
        print(f"Notificación creada: bucket={BUCKET_NAME} prefix={EVENT_DRIVEN_PREFIX} → topic={TOPIC_GCS}")
    except Exception as e:
        NOTIFICATION_DISPONIBLE = False
        print(f"  Error creando notificación: {str(e)[:200]}")
        print(f"  → Cambiando a modo SIMULADO.")
else:
    print(f"Modo SIMULADO: no se crea la notificación GCS→Pub/Sub.")
    print(f"El pipeline usará un evento sintético equivalente al que GCS publicaría.")

# 4. Crear subscription pull para consumir los eventos desde el notebook
#    Se crea siempre (independiente del modo): en real consume eventos de GCS,
#    en simulado la dejamos creada para mostrar el patrón completo.
sub_path = subscriber.subscription_path(PROJECT_ID, SUB_GCS)
try:
    subscriber.create_subscription(
        request={"name": sub_path, "topic": topic_path, "ack_deadline_seconds": 60}
    )
    print(f"Subscription creada: {SUB_GCS}")
except AlreadyExists:
    print(f"Subscription ya existe: {SUB_GCS}")

Modo SIMULADO: no se crea la notificación GCS→Pub/Sub.
El pipeline usará un evento sintético equivalente al que GCS publicaría.
Subscription creada: gcs-personio-consumer


---
## 3. Subir un archivo Personio y consumir el evento

Subimos una porción del CSV real `personio_data_v2_anon.csv` al prefijo monitorizado y consumimos el evento que GCS publica automáticamente.

In [5]:
# Preparar un CSV de muestra desde los datos reales
# Nota: en silver_personio.dim_employee la banda salarial está en `band` (no `position_band`).
sql_sample = f"""
SELECT
    employee_code,
    department,
    band AS position_band,
    office,
    country,
    hire_date,
    status
FROM `{PROJECT_ID}.silver_personio.dim_employee`
LIMIT 50
"""
df_sample = bq_client.query(sql_sample).to_dataframe()

# Convertir a CSV en memoria
csv_buf = StringIO()
df_sample.to_csv(csv_buf, index=False)

# Subir a GCS — esto disparará el evento OBJECT_FINALIZE
fecha_hoy = datetime.utcnow().strftime("%Y-%m-%d")
blob_path = f"{EVENT_DRIVEN_PREFIX}{fecha_hoy}/sample_personio_employees.csv"
blob = bucket.blob(blob_path)
blob.upload_from_string(csv_buf.getvalue(), content_type="text/csv")

print(f"Archivo subido: gs://{BUCKET_NAME}/{blob_path}")
print(f"Filas:          {len(df_sample)}")
print(f"Tamaño:         {blob.size} bytes")

Archivo subido: gs://project-9176af0b-ecb3-4050-859-datalake/event-driven-demo/2026-04-23/sample_personio_employees.csv
Filas:          50
Tamaño:         2959 bytes


In [6]:
# Consumir el evento desde la subscription (o simularlo si no hay notificación)
from google.api_core.exceptions import DeadlineExceeded

ultimo_evento = None

if NOTIFICATION_DISPONIBLE:
    # Nota: el primer evento tras crear la notificación puede tardar 30-60s en llegar.
    print("Esperando 5s para que GCS publique el evento...")
    time.sleep(5)

    received_messages = []
    MAX_INTENTOS = 4
    for intento in range(1, MAX_INTENTOS + 1):
        try:
            response = subscriber.pull(
                request={"subscription": sub_path, "max_messages": 10, "return_immediately": False},
                timeout=15.0,
            )
            received_messages = list(response.received_messages)
        except DeadlineExceeded:
            received_messages = []

        if received_messages:
            print(f"Recibidos {len(received_messages)} eventos en el intento {intento}")
            break
        else:
            if intento < MAX_INTENTOS:
                print(f"  Intento {intento}: aún sin mensajes, esperando 10s más...")
                time.sleep(10)
            else:
                print(f"  Sin eventos tras {MAX_INTENTOS} intentos.")

    ack_ids = []
    for received in received_messages:
        msg = received.message
        attrs = dict(msg.attributes)
        body = json.loads(msg.data.decode("utf-8"))

        print(f"\n  eventType:    {attrs.get('eventType')}")
        print(f"  bucket:       {attrs.get('bucketId')}")
        print(f"  object:       {attrs.get('objectId')}")
        print(f"  size:         {body.get('size')} bytes")
        print(f"  contentType:  {body.get('contentType')}")
        print(f"  publish_time: {msg.publish_time}")

        ack_ids.append(received.ack_id)
        ultimo_evento = body

    if ack_ids:
        subscriber.acknowledge(request={"subscription": sub_path, "ack_ids": ack_ids})
        print(f"\n  → {len(ack_ids)} eventos confirmados (ack)")

else:
    # --- MODO SIMULADO ---
    # Construimos el payload exacto que GCS publicaría para OBJECT_FINALIZE.
    # Esto permite que el pipeline de la siguiente celda consuma el mismo
    # formato de evento en clase que en producción.
    ultimo_evento = {
        "kind": "storage#object",
        "bucket": BUCKET_NAME,
        "name": blob_path,
        "size": str(blob.size),
        "contentType": "text/csv",
        "timeCreated": datetime.now(timezone.utc).isoformat(),
        "updated": datetime.now(timezone.utc).isoformat(),
    }
    attrs_sim = {
        "eventType": "OBJECT_FINALIZE",
        "bucketId": BUCKET_NAME,
        "objectId": blob_path,
        "payloadFormat": "JSON_API_V1",
    }
    print("Evento SIMULADO (payload idéntico al que GCS publicaría):")
    print(f"  eventType:    {attrs_sim['eventType']}")
    print(f"  bucket:       {attrs_sim['bucketId']}")
    print(f"  object:       {attrs_sim['objectId']}")
    print(f"  size:         {ultimo_evento['size']} bytes")
    print(f"  contentType:  {ultimo_evento['contentType']}")
    print(f"  timeCreated:  {ultimo_evento['timeCreated']}")

Evento SIMULADO (payload idéntico al que GCS publicaría):
  eventType:    OBJECT_FINALIZE
  bucket:       project-9176af0b-ecb3-4050-859-datalake
  object:       event-driven-demo/2026-04-23/sample_personio_employees.csv
  size:         2959 bytes
  contentType:  text/csv
  timeCreated:  2026-04-23T08:50:34.740426+00:00


---
## 4. Pipeline event-driven simulado: leer → transformar → cargar (idempotente)

Esta sección reproduce **lo que haría una Cloud Function disparada por Eventarc**, pero ejecutándose localmente para que sea inspeccionable celda a celda.

Patrón: **DELETE + INSERT por (`fecha_snapshot`, `source_file`)** = idempotencia. Reejecutar el pipeline nunca duplica filas.

In [7]:
# Crear la tabla destino en bronze (idempotente)
schema_raw = [
    bigquery.SchemaField("employee_code", "STRING"),
    bigquery.SchemaField("department", "STRING"),
    bigquery.SchemaField("position_band", "STRING"),
    bigquery.SchemaField("office", "STRING"),
    bigquery.SchemaField("country", "STRING"),
    bigquery.SchemaField("hire_date", "DATE"),
    bigquery.SchemaField("status", "STRING"),
    bigquery.SchemaField("fecha_snapshot", "DATE"),
    bigquery.SchemaField("source_file", "STRING"),
    bigquery.SchemaField("ingested_at", "TIMESTAMP"),
]
table_obj = bigquery.Table(TABLE_RAW, schema=schema_raw)
table_obj.time_partitioning = bigquery.TimePartitioning(
    type_=bigquery.TimePartitioningType.DAY, field="fecha_snapshot"
)
table_obj.clustering_fields = ["country", "department"]

try:
    bq_client.create_table(table_obj)
    print(f"Tabla creada:    {TABLE_RAW}")
except Exception as e:
    if "Already Exists" in str(e):
        print(f"Tabla ya existe: {TABLE_RAW}")
    else:
        raise

# Tabla de errores (dead letter)
schema_err = [
    bigquery.SchemaField("source_file", "STRING"),
    bigquery.SchemaField("row_data", "STRING"),
    bigquery.SchemaField("errores", "STRING"),
    bigquery.SchemaField("ingested_at", "TIMESTAMP"),
]
err_table = bigquery.Table(TABLE_ERRORS, schema=schema_err)
try:
    bq_client.create_table(err_table)
    print(f"Tabla creada:    {TABLE_ERRORS}")
except Exception as e:
    if "Already Exists" in str(e):
        print(f"Tabla ya existe: {TABLE_ERRORS}")
    else:
        raise

Tabla ya existe: project-9176af0b-ecb3-4050-859.bronze_personio.event_driven_raw
Tabla ya existe: project-9176af0b-ecb3-4050-859.bronze_personio.event_driven_errors


In [ ]:
def procesar_archivo_event_driven(bucket_name: str, file_name: str) -> dict:
    """
    Replica lo que haría una Cloud Function disparada por Eventarc:
      1. Leer archivo de GCS
      2. Parsear CSV
      3. Validar y transformar
      4. Cargar a BigQuery (idempotente)
    Devuelve métricas para logging.
    """
    t0 = time.time()
    metricas = {"file": file_name, "registros_leidos": 0, "registros_validos": 0, "registros_invalidos": 0}

    # 1. Leer
    blob = gcs_client.bucket(bucket_name).blob(file_name)
    contenido = blob.download_as_text(encoding="utf-8")

    # 2. Parsear
    df = pd.read_csv(StringIO(contenido))
    metricas["registros_leidos"] = len(df)

    # 3. Extraer fecha del path (formato: prefix/YYYY-MM-DD/...)
    fecha_match = re.search(r"(\d{4}-\d{2}-\d{2})", file_name)
    fecha = fecha_match.group(1) if fecha_match else datetime.now().strftime("%Y-%m-%d")

    df["fecha_snapshot"] = fecha
    df["source_file"] = file_name
    df["ingested_at"] = pd.Timestamp.now(tz="UTC")

    # Alinear dtypes con el esquema de la tabla destino (STRING/DATE/TIMESTAMP)
    # employee_code viene numérico del CSV y nuestro esquema lo trata como STRING.
    for col in ["employee_code", "department", "position_band", "office", "country", "status", "source_file"]:
        if col in df.columns:
            df[col] = df[col].astype("string")
    df["hire_date"] = pd.to_datetime(df["hire_date"], errors="coerce").dt.date
    df["fecha_snapshot"] = pd.to_datetime(df["fecha_snapshot"]).dt.date

    metricas["registros_validos"] = len(df)

    # 4. IDEMPOTENCIA: borrar lo que haya antes de cargar
    delete_sql = f"""
    DELETE FROM `{TABLE_RAW}`
    WHERE fecha_snapshot = '{fecha}' AND source_file = '{file_name}'
    """
    bq_client.query(delete_sql).result()

    # 5. Cargar
    job_config = bigquery.LoadJobConfig(write_disposition="WRITE_APPEND", schema=schema_raw)
    bq_client.load_table_from_dataframe(df, TABLE_RAW, job_config=job_config).result()

    metricas["duracion_ms"] = int((time.time() - t0) * 1000)
    return metricas

# Ejecutar el pipeline para el archivo que acabamos de subir
metricas = procesar_archivo_event_driven(BUCKET_NAME, blob_path)
print("Métricas del pipeline:")
for k, v in metricas.items():
    print(f"  {k}: {v}")

# Verificar idempotencia: ejecutar OTRA VEZ. Debe acabar con el mismo número de filas.
print("\n--- Reejecutando para probar idempotencia ---")
metricas2 = procesar_archivo_event_driven(BUCKET_NAME, blob_path)
print(f"  Reejecución completada en {metricas2['duracion_ms']}ms\n")

# Verificar el conteo
sql_count = f"""
SELECT COUNT(*) AS filas, COUNT(DISTINCT employee_code) AS empleados_unicos
FROM `{TABLE_RAW}`
WHERE source_file = '{blob_path}' AND fecha_snapshot = '{fecha_hoy}'
"""
df_check = bq_client.query(sql_count).to_dataframe()
print(f"Filas en BQ tras 2 ejecuciones: {df_check.iloc[0]['filas']} (esperado: {len(df_sample)})")
print(f"Empleados únicos: {df_check.iloc[0]['empleados_unicos']}")
print(f"Idempotencia OK" if df_check.iloc[0]['filas'] == len(df_sample) else "ALERTA: hay duplicados")

Métricas del pipeline:
  file: event-driven-demo/2026-04-23/sample_personio_employees.csv
  registros_leidos: 50
  registros_validos: 50
  registros_invalidos: 0
  duracion_ms: 4802

--- Reejecutando para probar idempotencia ---
  Reejecución completada en 5141ms

Filas en BQ tras 2 ejecuciones: 50 (esperado: 50)
Empleados únicos: 50
Idempotencia OK


---
## 5. Retry con backoff exponencial

Patrón estándar para llamadas a APIs externas o cargas a BigQuery: si falla, reintenta con espera creciente para no saturar el servicio fallido.

In [9]:
def retry_with_backoff(max_retries=3, base_delay=1, max_delay=30, exceptions=(Exception,)):
    """Decorador que reintenta una función con backoff exponencial."""
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            for attempt in range(max_retries + 1):
                try:
                    return func(*args, **kwargs)
                except exceptions as e:
                    if attempt == max_retries:
                        print(f"  Fallo definitivo tras {max_retries+1} intentos: {e}")
                        raise
                    delay = min(base_delay * (2 ** attempt), max_delay)
                    print(f"  Intento {attempt + 1} falló: {type(e).__name__} — reintentando en {delay}s...")
                    time.sleep(delay)
        return wrapper
    return decorator

# --- Demo: función que falla 2 veces y luego acierta ---
intentos = {"n": 0}

@retry_with_backoff(max_retries=4, base_delay=1)
def operacion_inestable():
    intentos["n"] += 1
    if intentos["n"] < 3:
        raise ConnectionError(f"Fallo simulado (intento {intentos['n']})")
    return f"Éxito en intento {intentos['n']}"

resultado = operacion_inestable()
print(f"\nResultado final: {resultado}")
print(f"Total intentos: {intentos['n']}")

  Intento 1 falló: ConnectionError — reintentando en 1s...
  Intento 2 falló: ConnectionError — reintentando en 2s...

Resultado final: Éxito en intento 3
Total intentos: 3


In [10]:
# --- Aplicado a una carga real a BigQuery ---
@retry_with_backoff(max_retries=3, base_delay=2, exceptions=(GoogleAPICallError, TimeoutError))
def cargar_a_bigquery_resiliente(df: pd.DataFrame, table_ref: str):
    """Carga un DataFrame a BigQuery con retry automático en errores transitorios."""
    job = bq_client.load_table_from_dataframe(
        df, table_ref,
        job_config=bigquery.LoadJobConfig(write_disposition="WRITE_APPEND", schema=schema_raw)
    )
    return job.result()

# Cargar de nuevo el sample (esta vez con retry blindado)
df_extra = df_sample.copy()
df_extra["fecha_snapshot"] = pd.to_datetime(fecha_hoy).date()
df_extra["source_file"] = blob_path + ".retry"
df_extra["ingested_at"] = pd.Timestamp.now(tz="UTC")
df_extra["hire_date"] = pd.to_datetime(df_extra["hire_date"], errors="coerce").dt.date
for col in ["employee_code", "department", "position_band", "office", "country", "status", "source_file"]:
    if col in df_extra.columns:
        df_extra[col] = df_extra[col].astype("string")

# Borrar primero (idempotencia)
bq_client.query(f"DELETE FROM `{TABLE_RAW}` WHERE source_file = '{blob_path}.retry'").result()

job = cargar_a_bigquery_resiliente(df_extra, TABLE_RAW)
print(f"\nCarga completada: {job.output_rows} filas insertadas")


Carga completada: 50 filas insertadas


---
## 6. Validación con split valid/invalid (dead letter)

Generamos un dataset con registros corruptos a propósito (employee_code vacío, salario fuera de rango, country inválido) y los separamos:
- **Válidos** → tabla principal
- **Inválidos** → tabla de errores (dead letter) con la razón del fallo

In [11]:
VALID_BANDS = {"P1", "P2", "P3", "E1", "E2", "M2", "M3"}

def validar_registro(row: dict) -> list:
    """Devuelve la lista de errores. Vacía si el registro es válido.

    Reglas alineadas con el formato real de silver_personio.dim_employee:
    - country se guarda como nombre completo ("Spain", "United States"), no ISO2.
    - position_band es opcional: puede ser NULL para empleados sin banda asignada.
    """
    errores = []

    ec = row.get("employee_code")
    if ec is None or pd.isna(ec) or str(ec).strip() == "":
        errores.append("employee_code_vacio")

    country = row.get("country")
    if country is None or pd.isna(country) or str(country).strip() == "":
        errores.append("country_vacio")

    band = row.get("position_band")
    # Solo validamos la banda si está presente (es un campo opcional en el Silver)
    if band is not None and not pd.isna(band) and str(band).strip() != "":
        band_str = str(band).upper().strip()
        if band_str not in VALID_BANDS:
            errores.append(f"position_band_invalida:{band_str}")

    return errores

# --- Construir un dataset mixto: 5 buenos + 4 corruptos ---
sample_buenos = df_sample.head(5).to_dict("records")

corruptos = [
    # 1) employee_code vacío
    {"employee_code": "", "department": "Engineering", "position_band": "P2", "office": "Madrid", "country": "Spain", "hire_date": "2025-01-15", "status": "Active"},
    # 2) position_band inválida
    {"employee_code": "EMP-X1", "department": "Sales", "position_band": "X9", "office": "Lima", "country": "Peru", "hire_date": "2025-02-01", "status": "Active"},
    # 3) MÚLTIPLES errores a la vez — la lista de errores los acumula todos
    {"employee_code": None, "department": "HR", "position_band": "INVALID_BAND", "office": "London", "country": None, "hire_date": "2024-08-10", "status": "Active"},
    # 4) country vacío
    {"employee_code": "EMP-X3", "department": "Engineering", "position_band": "P1", "office": "Madrid", "country": "", "hire_date": "2025-03-20", "status": "Active"},
]

mezcla = sample_buenos + corruptos
print(f"Dataset mixto: {len(mezcla)} registros ({len(sample_buenos)} buenos + {len(corruptos)} corruptos)\n")

# --- Split ---
validos, invalidos = [], []
for r in mezcla:
    errs = validar_registro(r)
    if errs:
        invalidos.append({
            "source_file": "demo-validacion",
            "row_data": json.dumps(r, default=str),
            "errores": ",".join(errs),
            "ingested_at": datetime.now(timezone.utc).isoformat(),
        })
    else:
        validos.append(r)

print(f"Válidos:   {len(validos)}")
print(f"Inválidos: {len(invalidos)}")
for inv in invalidos:
    print(f"  - errores: {inv['errores']}")

Dataset mixto: 9 registros (5 buenos + 4 corruptos)

Válidos:   5
Inválidos: 4
  - errores: employee_code_vacio
  - errores: position_band_invalida:X9
  - errores: employee_code_vacio,country_vacio,position_band_invalida:INVALID_BAND
  - errores: country_vacio


In [12]:
# Cargar inválidos a la tabla de errores
if invalidos:
    df_inv = pd.DataFrame(invalidos)
    df_inv["ingested_at"] = pd.to_datetime(df_inv["ingested_at"])
    bq_client.load_table_from_dataframe(
        df_inv, TABLE_ERRORS,
        job_config=bigquery.LoadJobConfig(write_disposition="WRITE_APPEND", schema=schema_err)
    ).result()
    print(f"  {len(df_inv)} registros inválidos enviados a {TABLE_ERRORS}")

# Verificar
sql_check_err = f"SELECT errores, COUNT(*) AS n FROM `{TABLE_ERRORS}` GROUP BY errores ORDER BY n DESC"
df_err_summary = bq_client.query(sql_check_err).to_dataframe()
print("\nResumen de errores acumulados:")
print(df_err_summary.to_string(index=False))

  4 registros inválidos enviados a project-9176af0b-ecb3-4050-859.bronze_personio.event_driven_errors

Resumen de errores acumulados:
                                                              errores  n
                          country_no_iso2,position_band_invalida:NONE 10
                                            position_band_invalida:X9  3
                                                  employee_code_vacio  3
                                                        country_vacio  3
                                                      country_no_iso2  2
employee_code_vacio,country_vacio,position_band_invalida:INVALID_BAND  1


---
## 7. Logging estructurado para Cloud Logging

Cuando una Cloud Function emite un `print` con un dict JSON en `stderr`, Cloud Logging lo parsea automáticamente y los campos quedan **filtrables y agregables** en el explorador.

Buenas prácticas:
- Severidad explícita (`INFO`, `WARNING`, `ERROR`)
- Contexto consistente (pipeline, archivo, fecha, registros, duración)
- Un identificador de ejecución (`run_id`) para trazar todos los logs de un mismo pipeline run

In [13]:
class StructuredLogHandler(logging.Handler):
    """Handler que emite cada log como una línea JSON. Cloud Logging lo parsea automáticamente."""
    EXTRA_KEYS = {"archivo", "registros", "fecha", "duracion_ms", "run_id", "tabla"}

    def emit(self, record):
        entry = {
            "severity": record.levelname,
            "message": record.getMessage(),
            "logger": record.name,
            "function": record.funcName,
        }
        for k, v in record.__dict__.items():
            if k in self.EXTRA_KEYS:
                entry[k] = v
        print(json.dumps(entry, default=str), file=sys.stderr)

logger = logging.getLogger("pipeline_event_driven")
logger.handlers.clear()
logger.addHandler(StructuredLogHandler())
logger.setLevel(logging.INFO)

# --- Ejecutar el pipeline emitiendo logs estructurados ---
import uuid
run_id = str(uuid.uuid4())[:8]

logger.info("Inicio pipeline", extra={"run_id": run_id, "archivo": blob_path, "fecha": fecha_hoy})

t0 = time.time()
metricas3 = procesar_archivo_event_driven(BUCKET_NAME, blob_path)
duracion = int((time.time() - t0) * 1000)

logger.info(
    "Carga completada",
    extra={
        "run_id": run_id,
        "archivo": blob_path,
        "fecha": fecha_hoy,
        "registros": metricas3["registros_validos"],
        "duracion_ms": duracion,
        "tabla": TABLE_RAW,
    }
)

if invalidos:
    logger.warning(
        "Registros inválidos enviados a dead letter",
        extra={
            "run_id": run_id,
            "registros": len(invalidos),
            "tabla": TABLE_ERRORS,
        }
    )

logger.info("Fin pipeline OK", extra={"run_id": run_id})

{"severity": "INFO", "message": "Inicio pipeline", "logger": "pipeline_event_driven", "function": "<module>", "run_id": "09146fc9", "archivo": "event-driven-demo/2026-04-23/sample_personio_employees.csv", "fecha": "2026-04-23"}
{"severity": "INFO", "message": "Carga completada", "logger": "pipeline_event_driven", "function": "<module>", "run_id": "09146fc9", "archivo": "event-driven-demo/2026-04-23/sample_personio_employees.csv", "fecha": "2026-04-23", "registros": 50, "duracion_ms": 4092, "tabla": "project-9176af0b-ecb3-4050-859.bronze_personio.event_driven_raw"}
{"severity": "WARNING", "message": "Registros inv\u00e1lidos enviados a dead letter", "logger": "pipeline_event_driven", "function": "<module>", "run_id": "09146fc9", "registros": 4, "tabla": "project-9176af0b-ecb3-4050-859.bronze_personio.event_driven_errors"}
{"severity": "INFO", "message": "Fin pipeline OK", "logger": "pipeline_event_driven", "function": "<module>", "run_id": "09146fc9"}


---
## 8. Limpieza idempotente

Borramos los recursos creados durante el notebook. Las tablas `event_driven_raw` y `event_driven_errors` quedan en `bronze_personio` para que puedas inspeccionarlas — bórralas manualmente si quieres limpieza completa.

In [14]:
# --- Borrar notificaciones del bucket (las creadas por este notebook) ---
for n in bucket.list_notifications():
    if n.topic_name == TOPIC_GCS and n.blob_name_prefix == EVENT_DRIVEN_PREFIX:
        n.delete()
        print(f"  Notificación borrada: id={n.notification_id}")

# --- Borrar subscription ---
try:
    subscriber.delete_subscription(request={"subscription": sub_path})
    print(f"  Subscription borrada: {SUB_GCS}")
except NotFound:
    print(f"  Subscription no existe: {SUB_GCS}")

# --- Borrar topic ---
try:
    publisher.delete_topic(request={"topic": topic_path})
    print(f"  Topic borrado:        {TOPIC_GCS}")
except NotFound:
    print(f"  Topic no existe:      {TOPIC_GCS}")

# --- Borrar archivos demo del bucket ---
borrados = 0
for blob_obj in bucket.list_blobs(prefix=EVENT_DRIVEN_PREFIX):
    blob_obj.delete()
    borrados += 1
print(f"  Archivos demo borrados: {borrados} en gs://{BUCKET_NAME}/{EVENT_DRIVEN_PREFIX}")

print("\nLimpieza completada.")

  Subscription borrada: gcs-personio-consumer
  Topic borrado:        gcs-personio-events
  Archivos demo borrados: 1 en gs://project-9176af0b-ecb3-4050-859-datalake/event-driven-demo/

Limpieza completada.


---

## Resumen

En este notebook hemos:
1. Configurado **notificaciones GCS → Pub/Sub** (alternativa runnable a Eventarc + Cloud Functions)
2. Subido un archivo Personio a GCS y **consumido el evento generado automáticamente**
3. Implementado un **pipeline event-driven idempotente** (DELETE+INSERT por fecha+fuente) — verificado que reejecutar no duplica
4. Aplicado **retry con backoff exponencial** a la carga BigQuery
5. **Validado registros** y separado válidos/inválidos con dead letter table
6. Emitido **logs JSON estructurados** legibles por Cloud Logging

### Migración a producción
Para llevarlo a producción reemplaza:
- **Subscriber pull en notebook** → Cloud Function (gen2) con trigger Eventarc del mismo evento
- **Logging a stderr** → ya funciona igual en Cloud Logging
- **Procesamiento síncrono** → desacoplar con Pub/Sub si el volumen lo requiere

El código de validación, idempotencia, retry y logging es **idéntico**: el mismo patrón runnable aquí se despliega tal cual en Cloud Functions.

### Siguiente sesión
**Sesión 3 (Módulos 5+6):** Orquestación profesional con Cloud Workflows / Composer y buenas prácticas avanzadas de BigQuery (particionado, clustering, stored procedures).